# Week 7 — Ensemble Methods: Voting, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost, and Imbalanced Data

**Course context:** A single model has a single point of view. Ensembles combine multiple models to get a more reliable prediction than any one of them alone — this is how most winning solutions in ML competitions (and a large share of production systems) actually work. This week goes from the simplest ensemble idea (voting) to the industry-standard gradient boosting libraries, and finishes with handling imbalanced datasets, a problem that shows up constantly in real classification tasks (fraud, churn, disease detection).

**Data file needed:** `Telco-Customer-Churn.csv` (Day 7), same folder as this notebook.

**How this notebook is organized:** one section per day, each with: what it covers → why it matters for AI work → code → pitfalls.


## Day 1 — Voting Classifiers: Combining Different Model Types

**What it covers:** Training three different classifiers (Logistic Regression, Decision Tree, KNN) individually, then combining their predictions with a `VotingClassifier`.

**Why it matters for AI work:** This is the simplest possible ensemble: majority vote. It works because different model types make **different kinds of mistakes** — a linear model and a tree-based model have different blind spots, so combining them often smooths out individual weaknesses. It's the intuition behind every more sophisticated ensemble method that follows this week.

**When to use what:**
- `voting='hard'` — majority vote on the predicted class labels (what most models predicted)
- `voting='soft'` — averages the predicted *probabilities* across models, then picks the highest. Usually performs slightly better than hard voting, but requires every model in the ensemble to support `predict_proba()`.

**Pitfalls:**
- A voting ensemble only helps if the underlying models are reasonably good AND make *different* kinds of errors. Combining three similar/weak models won't magically produce a strong one — "ensemble of bad models" is still bad.
- All three base models here were scaled and trained on the identical train/test split — for a completely fair comparison, that's correct, but note that Decision Trees don't actually need scaling (only Logistic Regression and KNN do). Scaling doesn't hurt trees, so a shared preprocessing pipeline like this is fine here.


In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

log_model = LogisticRegression()
dt_model = DecisionTreeClassifier(random_state=42)
knn_model = KNeighborsClassifier()

log_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)

ensemble_model = VotingClassifier(
    estimators=[("log_reg", log_model), ("decision_tree", dt_model), ("knn", knn_model)],
    voting="hard",
)
ensemble_model.fit(X_train, y_train)
y_pred_ensemble = ensemble_model.predict(X_test)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, log_model.predict(X_test)):.3f}")
print(f"Decision Tree Accuracy: {accuracy_score(y_test, dt_model.predict(X_test)):.3f}")
print(f"k-NN Accuracy: {accuracy_score(y_test, knn_model.predict(X_test)):.3f}")
print(f"Voting Ensemble Accuracy: {accuracy_score(y_test, y_pred_ensemble):.3f}")

Logistic Regression Accuracy: 1.000
Decision Tree Accuracy: 1.000
k-NN Accuracy: 1.000
Voting Ensemble Accuracy: 1.000


## Day 2 — Random Forest: Bagging Many Trees Together

**What it covers:** Random Forest — an ensemble of many decision trees, each trained on a random subset of data and features, with predictions averaged/voted across all of them. Plus tuning it with `GridSearchCV`.

**Why it matters for AI work:** Random Forest is one of the most reliable "just works" algorithms in ML — strong performance with minimal tuning, handles both numeric and categorical (once encoded) data, and gives you feature importances for free (Week 6 Day 4). It's a very reasonable default to try before reaching for anything fancier.

**How it reduces overfitting:** a single decision tree easily overfits (it can grow until every leaf is a single training example). Random Forest averages many such trees, each trained on a different random slice of the data (**bagging** = Bootstrap AGGregatING) and a random subset of features per split — the individual trees' errors tend to cancel out when averaged.

**Key hyperparameters:**
- `n_estimators` — number of trees. More trees = more stable predictions, at the cost of more computation. Diminishing returns past a certain point.
- `max_depth` — how deep each tree can grow. Deeper = more flexible but more prone to overfitting per-tree (though the forest averaging helps compensate).
- `max_features` — how many features each tree considers per split. Fewer = more randomness/diversity between trees (usually helps the ensemble), more = each tree is closer to a single "best" tree.

**Pitfalls:**
- `max_features='None'` (as a string) in a grid is a bug some people make — it should be the Python value `None`, not the string `"None"`. Passing the string silently gets treated by scikit-learn as "use all features" only if it recognizes it, otherwise it errors — always double check you're not accidentally quoting a value that should be a real Python object.
- Random Forest importances (Week 6 Day 4) are somewhat biased toward high-cardinality features — worth remembering when interpreting `.feature_importances_` from any model trained this week.


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Random Forest Accuracy: 0.9649122807017544

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [3]:
# Hyperparameter tuning with GridSearchCV
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", "log2", None],   # None as a real Python value, not the string "None"
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.3f}")

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 200}
Best Cross-Validation Accuracy: 0.963


## Day 3 — Gradient Boosting: Learning from Previous Mistakes

**What it covers:** `GradientBoostingClassifier` — an ensemble that builds trees **sequentially**, where each new tree tries to correct the errors of the ones before it, compared against Random Forest's parallel/independent tree-building.

**Why it matters for AI work:** Boosting is the family of techniques (Gradient Boosting → XGBoost → LightGBM → CatBoost, the rest of this week) that tends to win on structured/tabular data competitions. Understanding the core idea — sequential error correction — makes the more advanced libraries much less mysterious.

**Bagging (Random Forest) vs Boosting (Gradient Boosting) — the key difference:**
- **Bagging** — trees built independently/in parallel, in random subsets of data, then averaged. Reduces variance (overfitting).
- **Boosting** — trees built sequentially, each new tree focused on the previous ensemble's mistakes. Reduces bias (underfitting), but is more prone to overfitting if left unchecked (too many boosting rounds, or too high a learning rate) — this is why boosting has more hyperparameters to manage carefully than Random Forest.

**Key hyperparameters:**
- `learning_rate` — how much each new tree's correction counts. Lower = more conservative/stable, needs more trees (`n_estimators`) to compensate. This is the same learning-rate concept from Week 3's gradient descent, applied to boosting rounds instead of parameter updates.
- `n_estimators` — number of sequential boosting rounds (trees)
- `max_depth` — boosting trees are typically kept **shallow** (3-7 is common) — unlike Random Forest, where deeper trees are more often fine, since each boosting tree only needs to correct a small remaining error.

**Pitfalls:**
- Gradient Boosting trains **sequentially** — it can't parallelize tree-building the way Random Forest can, making it slower to train. This tradeoff (slower training, often better accuracy) is exactly what XGBoost/LightGBM (Days 4-5) are engineered to improve on.
- Too many boosting rounds with too high a learning rate is a fast route to overfitting — watch the gap between training and test accuracy, not just the test number alone.


In [4]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, y_pred_gb):.3f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.956

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [5]:
# Reduced grid vs the original (which had 3x3x3=27 combos x 5 folds = 135 fits) -
# same idea, faster to run here. Feel free to expand it back if you have time to spare.
param_grid = {
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
}

grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.3f}")

# Compare against Random Forest from Day 2, same data/split
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
print(f"\nRandom Forest Accuracy (for comparison): {accuracy_score(y_test, rf_model.predict(X_test)):.3f}")

Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation Accuracy: 0.965



Random Forest Accuracy (for comparison): 0.965


## Day 4 — XGBoost: Optimized, Regularized Gradient Boosting

**What it covers:** XGBoost (eXtreme Gradient Boosting) — a faster, more regularized implementation of the gradient boosting idea, using both the native `xgb.train` API and the scikit-learn-compatible `XGBClassifier`.

**Why it matters for AI work:** XGBoost has been one of the most consistently strong algorithms for structured/tabular data for years — a very common choice/baseline in industry and competitions. Knowing both APIs matters: the native API (`xgb.train` + `DMatrix`) is what you'll see in older code and performance-critical setups; `XGBClassifier` is what you'll use day-to-day since it plugs directly into scikit-learn's `Pipeline`/`GridSearchCV`.

**What XGBoost adds over plain Gradient Boosting:**
- Built-in **regularization** (controls model complexity directly, reducing overfitting risk beyond what `max_depth`/`learning_rate` alone provide)
- Much faster training (optimized, parallelized tree construction)
- Handles missing values natively (learns the best direction to send missing values at each split)

**Pitfalls:**
- `DMatrix` is XGBoost's own optimized data structure — you must convert your NumPy/Pandas data into it for the native API (`xgb.train`); `XGBClassifier` handles this conversion for you automatically.
- The full grid search below (`3 learning_rates × 3 n_estimators × 3 max_depth × 2 subsample × 2 colsample = 108 combinations × 5 folds = 540 model fits`) is genuinely slow. Trimmed here for a notebook that runs in reasonable time — this exact tradeoff (search space size vs compute budget) is the whole subject of Week 8.


In [6]:
import xgboost as xgb
from xgboost import XGBClassifier

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {"objective": "binary:logistic", "eval_metric": "logloss", "max_depth": 3, "eta": 0.1}
xgb_native_model = xgb.train(params, dtrain, num_boost_round=100)

y_pred_prob = xgb_native_model.predict(dtest)
y_pred = (y_pred_prob > 0.5).astype(int)   # convert predicted probabilities to class labels

print(f"XGBoost (native API) Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

XGBoost (native API) Accuracy: 0.956

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.93      0.94        43
           1       0.96      0.97      0.97        71

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [7]:
# scikit-learn-compatible API, with a trimmed grid search for reasonable runtime
xgb_clf = XGBClassifier(eval_metric="logloss", random_state=42)

param_grid = {
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
}

grid_search = GridSearchCV(estimator=xgb_clf, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.3f}")

gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)
print(f"\nPlain Gradient Boosting Accuracy (for comparison): {accuracy_score(y_test, gb_model.predict(X_test)):.3f}")

Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best Cross-Validation Accuracy: 0.965



Plain Gradient Boosting Accuracy (for comparison): 0.956


## Day 5 — LightGBM and CatBoost: Faster, and Native Categorical Handling

**What it covers:** Two more gradient boosting libraries, each with a specific advantage — LightGBM for speed on large datasets, CatBoost for handling categorical features natively without manual encoding.

**Why it matters for AI work:** These three libraries (XGBoost, LightGBM, CatBoost) are usually the shortlist for any serious tabular ML project — knowing their relative strengths lets you pick the right one instead of defaulting to whichever you learned first.

**When to use what:**
- **XGBoost** — well-established, reliable, huge community/documentation, a safe default
- **LightGBM** — noticeably faster on large datasets (uses a different, more efficient tree-growing strategy — "leaf-wise" instead of "level-wise"), good choice when training speed matters
- **CatBoost** — handles categorical features **natively** — you can pass the actual category strings/columns directly (via `cat_features=`) without one-hot or label encoding first, and it often needs less hyperparameter tuning to get decent results

**Pitfalls:**
- Passing already-**label-encoded** integer columns to CatBoost's `cat_features` (as the first CatBoost model below does, matching the original exercise) partly defeats the point — CatBoost's native categorical handling is designed to work directly on the original string/category values, not pre-converted integers. The second CatBoost model demonstrates the more correct usage: passing the true categorical columns unencoded.
- LightGBM can print verbose training logs by default — pass `verbose=-1` if you want quieter output, as done below.


In [8]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

features = ["Pclass", "Sex", "Age", "Fare", "Embarked"]
target = "Survived"

df.fillna({"Age": df["Age"].median()}, inplace=True)
df.fillna({"Embarked": df["Embarked"].mode()[0]}, inplace=True)

df_native = df.copy()   # keep an UNencoded copy for CatBoost's native categorical handling

label_encoders = {}
for col in ["Sex", "Embarked"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Data Shape: {X_train.shape}")
print(f"Test Data Shape: {X_test.shape}")

Training Data Shape: (712, 5)
Test Data Shape: (179, 5)


In [9]:
# LightGBM
lgb_model = lgb.LGBMClassifier(verbose=-1)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
print(f"LightGBM Accuracy: {accuracy_score(y_test, lgb_pred):.4f}")

# CatBoost on already label-encoded columns (matches original exercise, but see the note above)
cat_features_idx = ["Pclass", "Sex", "Embarked"]
cat_model = CatBoostClassifier(cat_features=cat_features_idx, verbose=0)
cat_model.fit(X_train, y_train)
cat_pred = cat_model.predict(X_test)
print(f"CatBoost (on encoded columns) Accuracy: {accuracy_score(y_test, cat_pred):.4f}")

# XGBoost, same data
xgb_model = XGBClassifier(eval_metric="logloss", random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")

LightGBM Accuracy: 0.8045


CatBoost (on encoded columns) Accuracy: 0.8156
XGBoost Accuracy: 0.7709


In [10]:
# CatBoost's actual native use case: pass the TRUE categorical (string) columns, unencoded
X_native = df_native[features]
y_native = df_native[target]
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(X_native, y_native, test_size=0.2, random_state=42)

cat_model_native = CatBoostClassifier(cat_features=["Sex", "Embarked"], verbose=0)
cat_model_native.fit(X_train_n, y_train_n)
cat_preds_native = cat_model_native.predict(X_test_n)
print(f"CatBoost (native, unencoded categoricals) Accuracy: {accuracy_score(y_test_n, cat_preds_native):.4f}")

CatBoost (native, unencoded categoricals) Accuracy: 0.8156


## Day 6 — Imbalanced Data: ROC-AUC and SMOTE

**What it covers:** Evaluating a classifier properly on a heavily imbalanced dataset (fraud detection style — very few positive cases among many negatives), using `class_weight` and SMOTE (Synthetic Minority Over-sampling) to help the model actually learn the minority class.

**Why it matters for AI work:** Imbalanced classification is extremely common in real applications: fraud (rare), disease diagnosis (rare), churn (often a minority), equipment failure (rare). A model that just predicts "not fraud" every time can score 99%+ accuracy on a 1%-fraud dataset while being completely useless — this is exactly why Week 5 Day 4 warned against trusting accuracy alone on imbalanced data.

**When to use what:**
- **`class_weight="balanced"`** — tells the model to penalize mistakes on the minority class more heavily during training. Free, built into most scikit-learn classifiers, a good first thing to try.
- **SMOTE** — generates *synthetic* new minority-class examples (not just duplicating existing ones) by interpolating between existing minority examples. Rebalances the training data directly.
- **ROC-AUC** — the right headline metric for imbalanced binary classification, since it evaluates ranking quality across all possible thresholds rather than a single accuracy number at a fixed 0.5 threshold.

**Pitfalls:**
- **Never apply SMOTE before splitting into train/test** (or apply it to the test set at all). SMOTE must be fit only on the **training** data, after the split — applying it before splitting leaks synthetic points derived from what should be unseen test data, inflating your reported performance. The code below applies it correctly (after the split, training set only).
- SMOTE assumes minority-class points are meaningfully close to each other in feature space for interpolation to make sense — it works less well on very high-dimensional or sparse data.


In [11]:
import numpy as np

# NOTE: the original exercise loaded a real credit-card-fraud dataset from an external host
# (storage.googleapis.com) that isn't reachable from every network. Simulating a dataset with
# the same shape (many numeric features, ~0.5% fraud rate) so this notebook is self-contained.
np.random.seed(42)
n_samples = 20000
n_features = 10
fraud_rate = 0.007

X_fraud = np.random.randn(n_samples, n_features)
y_fraud = np.zeros(n_samples, dtype=int)
n_fraud = int(n_samples * fraud_rate)
fraud_idx = np.random.choice(n_samples, n_fraud, replace=False)
y_fraud[fraud_idx] = 1
X_fraud[fraud_idx] += np.random.randn(n_fraud, n_features) * 2 + 3   # fraud cases shifted, to be somewhat separable

df_fraud = pd.DataFrame(X_fraud, columns=[f"V{i+1}" for i in range(n_features)])
df_fraud["Class"] = y_fraud

print("Class Distribution:\n", df_fraud["Class"].value_counts())
print(f"\nFraud rate: {df_fraud['Class'].mean():.4%}")

Class Distribution:
 Class
0    19860
1      140
Name: count, dtype: int64

Fraud rate: 0.7000%


In [12]:
from sklearn.metrics import roc_auc_score

X = df_fraud.drop(columns=["Class"])
y = df_fraud["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(random_state=42, class_weight="balanced")
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Classification Report (class_weight='balanced'):\n")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])
print(f"ROC-AUC: {roc_auc:.3f}")

Classification Report (class_weight='balanced'):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3972
           1       1.00      0.93      0.96        28

    accuracy                           1.00      4000
   macro avg       1.00      0.96      0.98      4000
weighted avg       1.00      1.00      1.00      4000

ROC-AUC: 1.000


In [13]:
from imblearn.over_sampling import SMOTE

# SMOTE applied ONLY to the training set, AFTER the split
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print("Class Distribution After SMOTE (training set only):\n", pd.Series(y_resampled).value_counts())

rf_model_smote = RandomForestClassifier(random_state=42)
rf_model_smote.fit(X_resampled, y_resampled)
y_pred_smote = rf_model_smote.predict(X_test)

print("\nClassification Report (SMOTE):\n")
print(classification_report(y_test, y_pred_smote))

roc_auc_smote = roc_auc_score(y_test, rf_model_smote.predict_proba(X_test)[:, 1])
print(f"ROC-AUC (SMOTE): {roc_auc_smote:.3f}")

Class Distribution After SMOTE (training set only):
 Class
0    15888
1    15888
Name: count, dtype: int64



Classification Report (SMOTE):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3972
           1       1.00      1.00      1.00        28

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000

ROC-AUC (SMOTE): 1.000


## Day 7 — Mini Project: Comparing Random Forest, XGBoost, and LightGBM on Customer Churn (with SMOTE)

**What it covers:** The full week's tools combined on a realistic dataset — encode, scale, split, SMOTE the training set, train three different ensemble models, and compare them side by side with both classification reports and ROC-AUC.

**Why it matters for AI work:** This is close to a template for a real "which model should we ship" comparison — the same shape of analysis you'd do before choosing a model for a production churn-prediction system.

**Pitfalls:**
- All three models are compared on the exact same train/test split and same SMOTE-resampled training data — that's important for a fair comparison. If each model saw a different split, differences in their scores could just be due to which rows they happened to get, not real differences in the algorithms.
- ROC-AUC and the classification report's precision/recall can tell different stories — a model can have a strong ROC-AUC (good ranking ability across all thresholds) but a weaker precision/recall at the default 0.5 threshold specifically. If the deployed system actually uses a fixed threshold, care about the metric that matches how it'll really be used.


In [14]:
df_telco = pd.read_csv("Telco-Customer-Churn.csv")
print("Dataset Info:")
df_telco.info()
print("\nClass Distribution:\n", df_telco["churn"].value_counts())

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      500 non-null    int64  
 1   gender           500 non-null    str    
 2   age              500 non-null    int64  
 3   tenure           500 non-null    int64  
 4   monthly_charges  500 non-null    float64
 5   total_charges    500 non-null    float64
 6   contract_type    500 non-null    str    
 7   payment_method   500 non-null    str    
 8   churn            500 non-null    int64  
dtypes: float64(2), int64(4), str(3)
memory usage: 35.3 KB

Class Distribution:
 churn
0    344
1    156
Name: count, dtype: int64


In [15]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

le = LabelEncoder()
for column in ["gender", "contract_type", "payment_method"]:
    df_telco[column] = le.fit_transform(df_telco[column])
df_telco["churn"] = le.fit_transform(df_telco["churn"])

scaler = StandardScaler()
numerical_features = ["tenure", "monthly_charges", "total_charges", "age"]
df_telco[numerical_features] = scaler.fit_transform(df_telco[numerical_features])

X = df_telco.drop(columns=["churn", "customer_id"])   # drop the ID column - see Week 5 Day 7's note on this
y = df_telco["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class Distribution After SMOTE:\n", pd.Series(y_train_resampled).value_counts())

Class Distribution After SMOTE:
 churn
1    275
0    275
Name: count, dtype: int64


In [16]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf_model.predict(X_test)
roc_auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

xgb_model = XGBClassifier(eval_metric="logloss", random_state=42)
xgb_model.fit(X_train_resampled, y_train_resampled)
y_pred_xgb = xgb_model.predict(X_test)
roc_auc_xgb = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])

lgb_model = LGBMClassifier(random_state=42, verbose=-1)
lgb_model.fit(X_train_resampled, y_train_resampled)
y_pred_lgb = lgb_model.predict(X_test)
roc_auc_lgb = roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1])

print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("XGBoost Report:\n", classification_report(y_test, y_pred_xgb))
print("LightGBM Report:\n", classification_report(y_test, y_pred_lgb))

print("ROC-AUC Scores:")
print(f"Random Forest: {roc_auc_rf:.3f}")
print(f"XGBoost: {roc_auc_xgb:.3f}")
print(f"LightGBM: {roc_auc_lgb:.3f}")

Random Forest Report:
               precision    recall  f1-score   support

           0       0.67      0.67      0.67        69
           1       0.26      0.26      0.26        31

    accuracy                           0.54       100
   macro avg       0.46      0.46      0.46       100
weighted avg       0.54      0.54      0.54       100

XGBoost Report:
               precision    recall  f1-score   support

           0       0.71      0.64      0.67        69
           1       0.34      0.42      0.38        31

    accuracy                           0.57       100
   macro avg       0.53      0.53      0.52       100
weighted avg       0.60      0.57      0.58       100

LightGBM Report:
               precision    recall  f1-score   support

           0       0.65      0.57      0.60        69
           1       0.25      0.32      0.28        31

    accuracy                           0.49       100
   macro avg       0.45      0.44      0.44       100
weighted avg    

## Week 7 Recap

| Day | Topic | Where you'll use it again |
|---|---|---|
| 1 | Voting classifier | The simplest ensemble idea |
| 2 | Random Forest (bagging) | A strong, low-effort default for tabular data |
| 3 | Gradient Boosting | The sequential-correction idea behind XGBoost/LightGBM/CatBoost |
| 4 | XGBoost | A common industry/competition default |
| 5 | LightGBM, CatBoost | Speed, and native categorical handling |
| 6 | SMOTE, ROC-AUC, class_weight | Any imbalanced classification problem |
| 7 | Full ensemble comparison mini-project | Realistic "which model should we ship" workflow |

**Before moving to Week 8:** you should be able to explain the difference between bagging and boosting in one or two sentences, and know why accuracy alone is misleading on imbalanced data. Week 8 goes deeper into hyperparameter tuning — the grids you trimmed for speed this week get the full, proper treatment (including smarter search strategies than brute-force grid search).
